# 02 - Embeddings (Grant Witness)
data source: grant-witness.us

### Imports

In [ ]:
# imports
import pandas as pd
import re
import os
from sentence_transformers import SentenceTransformer
import numpy as np

import torch
from transformers import AutoTokenizer


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df_epa = pd.read_csv('/content/drive/MyDrive/thesis/gw/01_epa.csv')
df_nih = pd.read_csv('/content/drive/MyDrive/thesis/gw/01_nih.csv')
df_nsf = pd.read_csv('/content/drive/MyDrive/thesis/gw/01_nsf.csv')
df_samhsa = pd.read_csv('/content/drive/MyDrive/thesis/gw/01_samhsa.csv')
df_cdc = pd.read_csv('/content/drive/MyDrive/thesis/gw/01_cdc.csv')

In [ ]:
# df_full = pd.read_csv('00_data/df_full.csv')
# df_full = pd.read_csv('/content/drive/MyDrive/thesis/gw/df_full.csv')

# Embeddings

In [ ]:
def pick_device():
    if torch.backends.mps.is_available():
        return 'mps'
    if torch.cuda.is_available():
        return 'cuda'
    return 'cpu'

device = pick_device()
print(device)

cuda


In [ ]:
dfs_abstract = {'nih': df_nih, 'nsf': df_nsf, 'epa': df_epa}
dfs_titles = {'nih': df_nih, 'nsf': df_nsf, 'epa': df_epa, 'samhsa': df_samhsa, 'cdc': df_cdc}

### nomic-embed-text-v1.5
https://huggingface.co/nomic-ai/nomic-embed-text-v1.5



In [ ]:
nomic_model = SentenceTransformer('nomic-ai/nomic-embed-text-v1.5', trust_remote_code=True, device=device)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/140 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/71.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

In [8]:
tok = nomic_model.tokenizer
sample = (df_nih['abstract'].fillna('').astype(str)
          .sample(min(5000, len(df_nih)), random_state=0).tolist())
lens = np.array([len(tok.encode(s)) for s in sample])
print(f'desc tokens — median {int(np.median(lens))}  p95 {int(np.percentile(lens,95))}  '
      f'p99 {int(np.percentile(lens,99))}  max {int(lens.max())}  >2048 {(lens>2048).mean():.1%}')

desc tokens — median 572  p95 714  p99 778  max 3184  >2048 0.0%


In [9]:
nomic_model.max_seq_length = 1024

### NIH

In [10]:
texts = df_nih['abstract'].fillna('').astype(str).tolist()
texts = [f"{'classification'}: {t}" for t in texts]
vectors = nomic_model.encode(texts, batch_size=128, show_progress_bar=True, normalize_embeddings=True, convert_to_numpy=True)

df_nih['abstract_embeddings_nomic'] = [v.tolist() for v in vectors]

titles = df_nih['project_title'].fillna('').astype(str).tolist()
titles = [f"{'classification'}: {t}" for t in titles]
vectors = nomic_model.encode(titles, batch_size=128, show_progress_bar=True, normalize_embeddings=True, convert_to_numpy=True)

df_nih['title_embeddings_nomic'] = [v.tolist() for v in vectors]

df_nih.to_parquet(f'/content/drive/MyDrive/thesis/gw/02_nih_nomic.parquet', index=False)


Batches:   0%|          | 0/46 [00:00<?, ?it/s]

[transformers] Detected the usage of `get_extended_attention_mask`: This function is deprecated and will be removed in v5.12.0. Please use the new API in `transformers.masking_utils`


Batches:   0%|          | 0/46 [00:00<?, ?it/s]

### NSF

In [12]:
texts = df_nsf['abstract'].fillna('').astype(str).tolist()
texts = [f"{'classification'}: {t}" for t in texts]
vectors = nomic_model.encode(texts, batch_size=128, show_progress_bar=True, normalize_embeddings=True, convert_to_numpy=True)

df_nsf['abstract_embeddings_nomic'] = [v.tolist() for v in vectors]

titles = df_nsf['project_title'].fillna('').astype(str).tolist()
titles = [f"{'classification'}: {t}" for t in titles]
vectors = nomic_model.encode(titles, batch_size=128, show_progress_bar=True, normalize_embeddings=True, convert_to_numpy=True)

df_nsf['title_embeddings_nomic'] = [v.tolist() for v in vectors]

df_nsf.to_parquet(f'/content/drive/MyDrive/thesis/gw/02_nsf_nomic.parquet', index=False)


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

### SAMHSA

In [13]:
texts = df_samhsa['abstract'].fillna('').astype(str).tolist()
texts = [f"{'classification'}: {t}" for t in texts]
vectors = nomic_model.encode(texts, batch_size=128, show_progress_bar=True, normalize_embeddings=True, convert_to_numpy=True)

df_samhsa['abstract_embeddings_nomic'] = [v.tolist() for v in vectors]

titles = df_samhsa['project_title'].fillna('').astype(str).tolist()
titles = [f"{'classification'}: {t}" for t in titles]
vectors = nomic_model.encode(titles, batch_size=128, show_progress_bar=True, normalize_embeddings=True, convert_to_numpy=True)

df_samhsa['title_embeddings_nomic'] = [v.tolist() for v in vectors]

df_samhsa.to_parquet(f'/content/drive/MyDrive/thesis/gw/02_samhsa_nomic.parquet', index=False)


Batches:   0%|          | 0/23 [00:00<?, ?it/s]

Batches:   0%|          | 0/23 [00:00<?, ?it/s]

### EPA

In [14]:
texts = df_epa['abstract'].fillna('').astype(str).tolist()
texts = [f"{'classification'}: {t}" for t in texts]
vectors = nomic_model.encode(texts, batch_size=128, show_progress_bar=True, normalize_embeddings=True, convert_to_numpy=True)

df_epa['abstract_embeddings_nomic'] = [v.tolist() for v in vectors]

titles = df_epa['project_title'].fillna('').astype(str).tolist()
titles = [f"{'classification'}: {t}" for t in titles]
vectors = nomic_model.encode(titles, batch_size=128, show_progress_bar=True, normalize_embeddings=True, convert_to_numpy=True)

df_epa['title_embeddings_nomic'] = [v.tolist() for v in vectors]

df_epa.to_parquet(f'/content/drive/MyDrive/thesis/gw/02_epa_nomic.parquet', index=False)


Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

### CDC

In [16]:
titles = df_cdc['project_title'].fillna('').astype(str).tolist()
titles = [f"{'classification'}: {t}" for t in titles]
vectors = nomic_model.encode(titles, batch_size=128, show_progress_bar=True, normalize_embeddings=True, convert_to_numpy=True)

df_cdc['title_embeddings_nomic'] = [v.tolist() for v in vectors]

df_cdc.to_parquet(f'/content/drive/MyDrive/thesis/gw/02_cdc_nomic.parquet', index=False)


Batches:   0%|          | 0/5 [00:00<?, ?it/s]